<a href="https://colab.research.google.com/github/andreagrioni/Tutorials/blob/master/03_BED_resize_and_extract_sequences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# resize bed file and extract sequences

### Jupyter notes

you can run system commands by:

```
! sys cmd
```

e.g. run unix ls command:

```
! ls
```


### we first need to install bedtools on the Virtual Machine

The bedtools documentation is available here:

https://bedtools.readthedocs.io/en/latest/content/installation.html

In [0]:
! apt-get install bedtools

In [0]:
! apt-get install pigz
! apt-get install samtools

### download target reference genome
run this code if you want to locally download a random target genome.

You can find a list of human and mouse genome at the GenCode repository:

https://www.gencodegenes.org/

e.g. URL of Human Genome GRCh38.p13 is:

ftp://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_32/GRCh38.primary_assembly.genome.fa.gz


After download, you will need to uncompress the file and index it with

pigz and samtools.


In [0]:
! wget ftp://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_32/GRCh38.primary_assembly.genome.fa.gz

In [0]:
! pigz -d GRCh38.primary_assembly.genome.fa.gz
! samtools faidx GRCh38.primary_assembly.genome.fa

## functions
we defined two main funtions, resize and bedtools_getfasta;

resize funtion will resize each bed region into an arbitrary windows.

bedtools_getfasta will extract genomic sequences of the resized bed file.


In [0]:
def resize(row, winsize=20, anchor='center'):
  """ the function resize the input coordiantes according to the 
      anchor definition; if anchor == 5 and strand == '+'
      the end coordinated with be extended as specified by winsize """

  start, end = row.start, row.end
  
  try:
    strand = row.strand
  except:
    strand = '5'

  if anchor == "5":
      if strand == "+":
        row.start = int(start)
        row.end = int(start) + int(winsize)
        
      elif strand == "-":
        row.start = int(end) - int(winsize)
        row.end = int(end)

      else:
          raise Exception("Error: Unknown strand in BED file")
  
  if anchor == "3":
  
      if strand == "+":
        row.start = int(end) - int(winsize)
        row.end = int(end)
  
      if strand == "-":
        row.start = int(start)
        row.end = int(start) + int(winsize)
  
      else:
        raise Exception("Error: Unknown strand in BED file")
  
  if anchor == "center":

      center = np.mean([start, end])
      half_dist = int(winsize) / int(2)
      row.start = int(center - half_dist)
      row.end = int(center + half_dist)
  
  return row

In [0]:
def bedtools_getfasta(reference, bed_input, output_name, opt=''):
    """ 
    fun runs bedtools getfasta on a pre-defined bed file,
    
    paramenters:
    reference=path to genome ref
    bed_input=path to bed input name
    output_name=output file name
    opt=additional bedtools options (see bedtools documentation)
    """
    cmd = f'bedtools getfasta {opt} -fi {reference} -bed {bed_input} > {output_name}'
    
    output = subprocess.run(cmd, shell=True, encoding="utf-8", stdout=subprocess.PIPE)
    
    return output_name

## Main script
You will need to hard-code the script arguments, such as:

reference=path to genome reference file
bed_file=path to bed file name

options:
bed_file_name_out = path to new bed file name with updated coordinates
output_name = path to fasta output file name with sequences.

## define script arguments

In [0]:
# settings and arguments
import pandas as pd
import io
import random
import subprocess
import copy
import os
import numpy as np

# script arguments
reference = str() or 'GRCh38.primary_assembly.genome.fa'
bed_file = str() # bed file name
bed_file_name_out = str()
output_name = str()


# this code will create a test bed file if not provided
if not bed_file_name_out and bed_file:
  bed_file_name_out=bed_file.replace('.bed', '.updated.bed')
elif not bed_file_name_out and not bed_file:
  bed_file_name_out='test.updated.bed'

if not output_name:
  output_name = bed_file_name_out.replace('.bed', '.fasta')

if not bed_file:
  print('run test')
  random_bed = str()
  for i in range(0,20):
    start = random.randint(10, 20)
    end = start + 50
    random_bed += f'chr{random.randint(1, 22)}\t{start}\t{end}\n'
  bed_file = io.StringIO(random_bed)

# check if bedtools is available in the environment
test_bedtools = subprocess.run(
    'bedtools --help',
    shell=True,
    encoding="utf-8",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
    )

if not test_bedtools.stdout:
    print('bedtools not available')

## run script

In [0]:
# main script
if test_bedtools.stdout:
  bed_file_df = pd.read_csv(
      bed_file, sep='\t'
  )

  if bed_file_df.shape[1] == 3:
    bed_file_df.columns = ['chrom', 'start', 'end']
  elif bed_file_df.shape[1] == 6:
    bed_file_df.columns = ['chrom', 'start', 'end', 'name', 'score', 'strad']

  # add column distance == interval size
  bed_file_df['distance'] = bed_file_df['end'] - bed_file_df['start']

  # safety check: reverse start-end coordinates where distance < 0
  bed_file_df.loc[bed_file_df['distance'] < 0,
              ['start', 'end']] = bed_file_df.loc[
                                                  bed_file_df['distance'] < 0,
                                                  ['end', 'start']
                                                  ].values
  # resize coordinates
  bed_file_df.apply(resize, winsize=20, anchor='center', axis=1)
  bed_file_df.drop(['distance'], axis=1, inplace=True)
  bed_file_df['start'] = bed_file_df['start'].astype(int)
  bed_file_df['end'] = bed_file_df['end'].astype(int)
  # write updated bed file
  bed_file_df.to_csv(
      bed_file_name_out,
      sep='\t',
      header=False,
      index=False
      )

## extract sequences

In [0]:
# get sequences
bedtools_getfasta(reference, bed_file_name_out, output_name, opt='-tab')

### check output

In [0]:
pd.read_csv(output_name, sep='\t', names=['region', 'sequence']).head(10)